# Week 15: Georeferencing GOES and parallax (Capstone 3 week)

**Track:** Remote Sensing Specialist (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/15/](https://launchdetect.com/academy/week/15/)  
**Track index:** [https://launchdetect.com/academy/remote-sensing-specialist/](https://launchdetect.com/academy/remote-sensing-specialist/)

---

_Track 3 culminates here: a GOES hotspot at (x, y) in the fixed grid is NOT directly at the (lat, lon) directly below — parallax matters, especially for tall plumes. This week is the math, and the capstone is a working thermal plume detector._


## Why this week matters

Without parallax correction, a plume detection at GOES limb can be 25 km off the actual launch site. Capstone 3 evaluates within ±5 km, so this week is the difference between a passing and failing capstone.


## Learning objectives

By the end of this lab you will be able to:

- Convert ABI fixed-grid coordinates to lat/lon
- Apply parallax correction for high-altitude plumes
- Handle the GOES 'view from the equator' geometry
- Account for limb effects at the edge of the disk


## Setup — and why these dependencies

This lab uses: `leafmap[common] xarray rasterio netCDF4 satpy`. Run the cell below once; Colab installs into the runtime.


In [ ]:
# Install required packages
!pip install -q leafmap[common] xarray rasterio netCDF4 satpy


## The approach (and why)

Implement the GOES fixed-grid-to-(lat, lon) projection from the official Product Definition Document. Test against the known sub-satellite point. Add parallax correction afterward.


In [ ]:
# Week 15: georeference GOES fixed-grid pixel to (lat, lon) + parallax correction.
# CAPSTONE 3 START.
import numpy as np

# GOES-R fixed-grid projection constants (from the GOES-R Product Definition Document)
H = 35786023.0 + 6378137.0   # satellite altitude + Earth equatorial radius (m)
lon_origin_rad = np.radians(-137.0)  # GOES-18 sub-satellite longitude (operational since 2023-01)
r_eq = 6378137.0
r_pol = 6356752.31414

def fixed_grid_to_latlon(x_rad: float, y_rad: float) -> tuple[float, float]:
    """Convert GOES fixed-grid scan/elevation angles to (lat, lon) in degrees."""
    a = np.sin(x_rad)**2 + (np.cos(x_rad)**2 * (np.cos(y_rad)**2 + (r_eq/r_pol)**2 * np.sin(y_rad)**2))
    b = -2 * H * np.cos(x_rad) * np.cos(y_rad)
    c = H**2 - r_eq**2
    r_s = (-b - np.sqrt(b**2 - 4*a*c)) / (2*a)
    s_x = r_s * np.cos(x_rad) * np.cos(y_rad)
    s_y = -r_s * np.sin(x_rad)
    s_z = r_s * np.cos(x_rad) * np.sin(y_rad)
    lat = np.degrees(np.arctan2((r_eq/r_pol)**2 * s_z, np.sqrt((H - s_x)**2 + s_y**2)))
    lon = np.degrees(lon_origin_rad + np.arctan2(s_y, H - s_x))
    return lat, lon

# Test: pixel at scan=0, elevation=0 should be at the sub-satellite point
lat, lon = fixed_grid_to_latlon(0.0, 0.0)
print(f"Sub-satellite point: ({lat:.4f}, {lon:.4f}) — expected (0, -137.0)")

# TODO: implement parallax correction for plumes at 50 km altitude,
# then build the full Capstone 3 plume detector that outputs
# (timestamp, lat, lon, T_b, area_km²) records.

## What just happened — and why it works

GOES doesn't produce lat/lon imagery natively — it produces scan/elevation angle imagery. The conversion is a closed-form geometric calculation that intersects the line-of-sight from satellite to pixel with the WGS84 ellipsoid. The math looks intimidating but is just ellipsoid-intersection from the GOES PDD.


## Common gotchas

- The (r_eq/r_pol) ratio matters — don't simplify to a sphere.
- For ABI 'sweep_x_axis' you're projecting onto the equatorial plane; double-check sign conventions against your specific data.
- Parallax correction assumes you know the plume altitude. Plume altitude varies from 0 to 80 km over the boost phase. Use a reasonable average (e.g., 30-50 km) or correct per detected frame's expected altitude.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/15/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
